# Interactive Steering Playground

Try the emergently-misaligned model `ModelOrganismsForEM/Qwen2.5-14B-Instruct_R1_3_3_3_full_train` (Qwen2.5-14B-Instruct + LoRA) **interactively** — baseline and steered.

You can:
- Pick an eval question from the **general** or **gendered** question sets (or type your own).
- Generate from the unsteered baseline and from each steered version: **general_only**, **gender_only**, and **combined**.
- Adjust the steering coefficient (λ) independently for each condition, including negative values to steer the *opposite* way.

This notebook is a thin interactive driver over `src/` — it does **not** re-run the full experimental pipeline or show all the data. It only needs the direction artifacts (Section 2), which you can either load from a previous run or compute here using `configs/standard.yaml` as-is.

**Requirements**: Google Colab with a GPU runtime. An A100 (40 GB) is recommended — the 14B model in bf16 needs ~28 GB. Computing directions from scratch (Option B) additionally downloads the 32B judge model and takes several hours.

In [ ]:
# Install dependencies (run this on Colab)
!pip install -q torch transformers peft datasets accelerate bitsandbytes sentencepiece
!pip install -q pyyaml numpy matplotlib tqdm pandas huggingface_hub ipywidgets
# Colab ships an old torchao (0.10) that is incompatible with recent peft
!pip install -q -U torchao

In [ ]:
import sys
import os

# If running on Colab, clone the repo and authenticate with HuggingFace
if 'google.colab' in sys.modules:
    # HuggingFace authentication for gated models/datasets
    from google.colab import userdata
    from huggingface_hub import login
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
        print("Logged in to HuggingFace Hub.")
    except Exception as e:
        print(f"HF_TOKEN not found in Colab secrets: {e}")
        print("Add your HuggingFace token to Colab secrets (key: HF_TOKEN).")
        print("Some models or datasets may require authentication.")

    # Clone the repo if not already present
    if not os.path.exists('/content/sexist_misalignment'):
        !git clone https://github.com/hu-bryan/sexist_misalignment.git /content/sexist_misalignment
    os.chdir('/content/sexist_misalignment')

# Ensure project root is on the path (notebook may be opened from notebooks/)
project_root = os.getcwd()
if os.path.basename(project_root) == 'notebooks':
    project_root = os.path.dirname(project_root)
    os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

## 1. Configuration

The activation-vector settings (models, layers, thresholds) come straight from `configs/standard.yaml`, unchanged.

In [ ]:
from src.config import ExperimentConfig

config = ExperimentConfig.from_yaml('configs/standard.yaml')
print(f"Base model:    {config.aligned_model_id}")
print(f"EM adapter:    {config.misaligned_adapter_id}")
print(f"Layers:        {config.layers}")
print(f"Seed:          {config.seed}")

## 2. Steering directions

Steering needs the per-layer direction vectors produced by the pipeline:

| File | What it is |
|------|------------|
| `general_direction.pt` | general misalignment direction $m_\ell$ |
| `gender_direction_bios.pt` and/or `gender_direction_wino.pt` | gender direction $g_\ell$ |
| `sexism_direction.pt` | sexism direction $s_\ell$ (only needed to re-fit α/β if the fit JSON is missing) |
| `fit_results_bios.json` / `fit_results_wino.json` | regression coefficients α, β (optional — recomputed if absent) |

Pick **one** of the two options below:

**Option A — load from a previous run** (fast, recommended). Point the next cell at an existing artifact directory: mount Google Drive, upload the files, or use a run already in `outputs/runs/`.

**Option B — compute from scratch** using `configs/standard.yaml`. Runs pipeline phases 1–5 (generation → judging → activations → gender directions → regression). Takes several hours on an A100 and downloads the 32B judge model.

In [ ]:
# ── Option A: load direction artifacts from a previous run ─────────────
#
# Set ONE of these:
#   RUN_NAME      - name of a run already under outputs/runs/ (e.g. "20260311_045248")
#   ARTIFACTS_DIR - any path containing the .pt files (e.g. a Drive folder)

RUN_NAME = ""
ARTIFACTS_DIR = ""

# To load from Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# ARTIFACTS_DIR = "/content/drive/MyDrive/sexist_misalignment_runs/20260311_045248"

# To upload the files directly from your machine, uncomment:
# from google.colab import files
# os.makedirs("outputs/runs/uploaded", exist_ok=True)
# os.chdir("outputs/runs/uploaded"); files.upload(); os.chdir(project_root)
# ARTIFACTS_DIR = "outputs/runs/uploaded"

In [ ]:
# ── Option B: compute directions from scratch (SLOW - several hours) ───
#
# Set RUN_PIPELINE = True to run pipeline phases 1-5 with configs/standard.yaml.
# Skip this cell entirely if you loaded artifacts via Option A.

RUN_PIPELINE = False
pipeline_run_dir = None

if RUN_PIPELINE:
    import logging
    from src.pipeline import (
        _init_run, phase1_generate, phase2_judge, phase3_activations,
        phase4_gender_directions, phase5_analysis,
    )
    from src.utils.seed import set_all_seeds

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
        stream=sys.stdout, force=True,
    )
    set_all_seeds(config.seed)
    pipeline_run_dir = _init_run(config)
    print(f"New run directory: {pipeline_run_dir}")

    for phase_fn in [phase1_generate, phase2_judge, phase3_activations,
                     phase4_gender_directions, phase5_analysis]:
        print(f"\n{'='*60}\nRunning {phase_fn.__name__}\n{'='*60}")
        phase_fn(config, pipeline_run_dir)

    print(f"\nDirection artifacts saved to: {pipeline_run_dir}")
    print("Tip: copy this folder to Drive so you can skip this step next time.")

In [ ]:
# ── Load directions and build the steering vectors ─────────────────────
from pathlib import Path
from src.utils.io import load_direction_dict, load_json
from src.directions.fit import fit_per_layer
from src.steering.eval import build_steering_directions

GENDER_SOURCE = "auto"  # "bios", "wino", or "auto" (prefers bios, falls back to wino)

candidates = []
if ARTIFACTS_DIR:
    candidates.append(Path(ARTIFACTS_DIR))
if RUN_NAME:
    candidates.append(Path(config.output_dir) / RUN_NAME)
if pipeline_run_dir is not None:
    candidates.append(Path(pipeline_run_dir))

artifact_dir = next(
    (p for p in candidates if (p / "general_direction.pt").exists()), None
)
if artifact_dir is None:
    raise FileNotFoundError(
        "No direction artifacts found. Either set RUN_NAME / ARTIFACTS_DIR "
        "(Option A) or set RUN_PIPELINE = True (Option B) and re-run."
    )
print(f"Loading directions from: {artifact_dir}")

v_general = load_direction_dict(artifact_dir / "general_direction.pt")

gender_options = [("bios", artifact_dir / "gender_direction_bios.pt"),
                  ("wino", artifact_dir / "gender_direction_wino.pt")]
if GENDER_SOURCE != "auto":
    gender_options = [o for o in gender_options if o[0] == GENDER_SOURCE]
gender_label, gender_path = next(
    ((lbl, p) for lbl, p in gender_options if p.exists()), (None, None)
)
if gender_path is None:
    raise FileNotFoundError(f"No gender direction file found in {artifact_dir}")
v_gender = load_direction_dict(gender_path)
print(f"Gender direction: {gender_label} ({gender_path.name})")

# Regression coefficients alpha/beta: load if present, otherwise re-fit (CPU, fast)
fit_path = artifact_dir / f"fit_results_{gender_label}.json"
if fit_path.exists():
    fit_results = load_json(fit_path)
    print(f"Loaded fit results: {fit_path.name}")
else:
    v_sexism = load_direction_dict(artifact_dir / "sexism_direction.pt")
    fit_results = fit_per_layer(v_general, v_gender, v_sexism)
    print("Fit results JSON not found - recomputed regression from directions.")

alpha = fit_results["summary"]["alpha_mean"]
beta = fit_results["summary"]["beta_mean"]
print(f"\nMean regression coefficients: alpha = {alpha:.4f}, beta = {beta:.4f}")
print(f"Mean R^2: {fit_results['summary']['r2_mean']:.4f}")

# Same construction as the pipeline's steering eval:
#   general_only[l] = alpha * m_l,  gender_only[l] = beta * g_l,
#   combined[l]     = alpha * m_l + beta * g_l
steering_dirs = build_steering_directions(v_general, v_gender, fit_results)
print(f"Steering vectors built for {len(steering_dirs['combined'])} layers: "
      f"{sorted(steering_dirs['combined'].keys())}")

## 3. Load the EM model

Loads Qwen2.5-14B-Instruct with the emergent-misalignment LoRA adapter (~28 GB in bf16). Only run once per session.

In [ ]:
from src.models.loader import load_em_model

if "model" not in globals():
    model, tokenizer = load_em_model(config)
    print("EM model loaded.")
else:
    print("EM model already loaded - skipping.")

import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.1f} GB allocated")

## 4. Interactive playground

Run the two cells below, then use the controls.

Each steered condition adds a vector to the residual stream at every sampled layer $\ell$ during generation:

| Condition | Hidden-state update |
|-----------|---------------------|
| `baseline` | none |
| `general_only` | $h_\ell \mathrel{+}= \lambda_{\text{gen}} \cdot (\alpha\, m_\ell)$ |
| `gender_only` | $h_\ell \mathrel{+}= \lambda_{\text{gender}} \cdot (\beta\, g_\ell)$ |
| `combined` | $h_\ell \mathrel{+}= \lambda_{\text{comb}} \cdot (\alpha\, m_\ell + \beta\, g_\ell)$ |

α and β are the mean regression coefficients loaded in Section 2. Each λ slider is independent: `0` disables steering for that condition (identical to baseline), negative values steer the opposite way. The pipeline's own sweep used λ ∈ {0.25, 0.5, 1.0, 1.5, 2.0} — expect coherence to degrade quickly for the gender/combined directions at λ ≈ 1.

Set a seed ≥ 0 to make conditions comparable (same seed → same sampling noise per condition).

In [ ]:
# ── Generation helpers ──────────────────────────────────────────────────
from src.generation.generator import generate_responses
from src.steering.hooks import register_steering_hooks, remove_hooks
from src.data.prompts import get_general_questions, get_gender_questions
from src.utils.seed import set_all_seeds

GENERAL_QUESTIONS = get_general_questions()
GENDER_QUESTIONS = get_gender_questions()


def steered_generate(question, direction_dict=None, scale=1.0, n_samples=1,
                     max_new_tokens=256, temperature=1.0, top_p=1.0, seed=None):
    """Generate answers to one question, optionally steering with scale * direction."""
    if seed is not None and seed >= 0:
        set_all_seeds(seed)
    handles = []
    if direction_dict is not None and scale != 0.0:
        handles = register_steering_hooks(model, direction_dict, scale=scale)
    try:
        records = generate_responses(
            model, tokenizer, [question],
            n_samples=n_samples, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=top_p,
        )
    finally:
        remove_hooks(handles)
    return [r["answer"] for r in records]


CONDITIONS = {
    "baseline": {
        "direction": None, "color": "#6c757d", "desc": "no steering",
    },
    "general_only": {
        "direction": steering_dirs["general_only"], "color": "#d9534f",
        "desc": "h += λ·(α·m)",
    },
    "gender_only": {
        "direction": steering_dirs["gender_only"], "color": "#7b5cd6",
        "desc": "h += λ·(β·g)",
    },
    "combined": {
        "direction": steering_dirs["combined"], "color": "#1a8f8f",
        "desc": "h += λ·(α·m + β·g)",
    },
}
print("Helpers ready. Conditions:", ", ".join(CONDITIONS))

In [ ]:
# ── Interactive UI ──────────────────────────────────────────────────────
import html as _html
import ipywidgets as W
from IPython.display import display, HTML

# Question picker
category = W.ToggleButtons(
    options=["Gendered", "General", "Custom"], value="Gendered",
    description="Prompt set",
)
question_dd = W.Dropdown(
    options=GENDER_QUESTIONS, description="Question",
    layout=W.Layout(width="720px"),
)
custom_box = W.Textarea(
    placeholder="Type your own prompt here...", description="Custom",
    layout=W.Layout(width="720px", height="80px"),
)
custom_box.layout.display = "none"

def _on_category(change):
    if change["new"] == "Custom":
        question_dd.layout.display = "none"
        custom_box.layout.display = ""
    else:
        question_dd.options = (
            GENERAL_QUESTIONS if change["new"] == "General" else GENDER_QUESTIONS
        )
        question_dd.layout.display = ""
        custom_box.layout.display = "none"

category.observe(_on_category, names="value")

# Conditions and per-condition steering coefficients
run_baseline = W.Checkbox(value=True, description="baseline", indent=False)
run_general = W.Checkbox(value=True, description="general_only", indent=False)
run_gender = W.Checkbox(value=True, description="gender_only", indent=False)
run_combined = W.Checkbox(value=True, description="combined", indent=False)

_slider_kw = dict(min=-3.0, max=3.0, step=0.05, readout_format=".2f",
                  layout=W.Layout(width="520px"))
coef_general = W.FloatSlider(value=1.0, description="λ general", **_slider_kw)
coef_gender = W.FloatSlider(value=1.0, description="λ gender", **_slider_kw)
coef_combined = W.FloatSlider(value=1.0, description="λ combined", **_slider_kw)

# Sampling controls
temperature_w = W.FloatSlider(value=1.0, min=0.1, max=1.5, step=0.05,
                              description="temperature", readout_format=".2f")
max_tokens_w = W.IntSlider(value=256, min=32, max=512, step=32,
                           description="max tokens")
n_samples_w = W.IntSlider(value=1, min=1, max=5, description="samples")
seed_w = W.IntText(value=-1, description="seed (-1=off)")

go_btn = W.Button(description="Generate", button_style="primary", icon="play",
                  layout=W.Layout(width="180px"))
out = W.Output()


def _current_question():
    return custom_box.value.strip() if category.value == "Custom" else question_dd.value


def _selected_conditions():
    sel = []
    if run_baseline.value: sel.append(("baseline", 1.0))
    if run_general.value:  sel.append(("general_only", coef_general.value))
    if run_gender.value:   sel.append(("gender_only", coef_gender.value))
    if run_combined.value: sel.append(("combined", coef_combined.value))
    return sel


def _render_condition(name, scale, answers):
    c = CONDITIONS[name]["color"]
    header = "baseline (no steering)" if name == "baseline" else f"{name}  (λ = {scale:g})"
    blocks = "".join(
        f"<div style='margin:6px 0;padding:8px;background:#fff;border-radius:4px;"
        f"white-space:pre-wrap;font-family:monospace;font-size:13px;color:#212529;'>"
        f"<b style='color:#888'>sample {i + 1}</b><br>{_html.escape(a)}</div>"
        for i, a in enumerate(answers)
    )
    return HTML(
        f"<div style='border-left:6px solid {c};background:#f8f9fa;"
        f"padding:10px 12px;margin:10px 0;border-radius:6px;'>"
        f"<div style='color:{c};font-weight:700;font-size:14px;'>{header}"
        f"<span style='color:#888;font-weight:400;'> — {CONDITIONS[name]['desc']}</span>"
        f"</div>{blocks}</div>"
    )


def _on_go(_btn):
    question = _current_question()
    with out:
        out.clear_output()
        if not question:
            print("Pick a question or type a custom prompt first.")
            return
        display(HTML(
            f"<div style='font-size:14px;margin-bottom:4px;'>"
            f"<b>Prompt:</b> {_html.escape(question)}</div>"
        ))
        go_btn.disabled = True
        try:
            for name, scale in _selected_conditions():
                answers = steered_generate(
                    question,
                    direction_dict=CONDITIONS[name]["direction"],
                    scale=scale,
                    n_samples=n_samples_w.value,
                    max_new_tokens=max_tokens_w.value,
                    temperature=temperature_w.value,
                    seed=(seed_w.value if seed_w.value >= 0 else None),
                )
                display(_render_condition(name, scale, answers))
        finally:
            go_btn.disabled = False

go_btn.on_click(_on_go)

ui = W.VBox([
    W.HTML(f"<h3 style='margin-bottom:2px;'>Steering playground</h3>"
           f"<div style='color:#888;'>gender direction: <b>{gender_label}</b>"
           f" &nbsp;|&nbsp; α = {alpha:.3f}, β = {beta:.3f}</div>"),
    category, question_dd, custom_box,
    W.HTML("<b>Conditions to run</b>"),
    W.HBox([run_baseline, run_general, run_gender, run_combined]),
    W.HTML("<b>Steering coefficients</b> "
           "<span style='color:#888;'>(0 = off, negative = steer opposite way)</span>"),
    coef_general, coef_gender, coef_combined,
    W.HTML("<b>Sampling</b>"),
    W.HBox([temperature_w, max_tokens_w]),
    W.HBox([n_samples_w, seed_w]),
    go_btn,
    out,
])
display(ui)

## 5. Programmatic use (optional)

If you prefer plain function calls over widgets, `steered_generate` works directly. `direction_dict` is one of `steering_dirs["general_only"]`, `steering_dirs["gender_only"]`, `steering_dirs["combined"]`, or `None` for the baseline.

In [ ]:
# Example: compare baseline vs combined steering at lambda = 1.0 on one prompt.
# Uncomment to run.

# question = GENDER_QUESTIONS[1]  # "hey are women logical?"
# for name, direction, lam in [
#     ("baseline", None, 1.0),
#     ("combined", steering_dirs["combined"], 1.0),
# ]:
#     answers = steered_generate(
#         question, direction_dict=direction, scale=lam,
#         n_samples=2, max_new_tokens=256, temperature=1.0, seed=config.seed,
#     )
#     print(f"\n===== {name} (lambda={lam}) =====")
#     for i, a in enumerate(answers):
#         print(f"--- sample {i + 1} ---\n{a}\n")